In [ ]:
import csv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
con =sqlite3.connect("../db.sqlite3")
cur = con.cursor()
res = cur.execute("""
    SELECT
        restaurant_order.id,
        restaurant_order.datetime,
        restaurant_orderitem.quantity,
        restaurant_product.price,
        restaurant_product.name
    FROM restaurant_order
    INNER JOIN restaurant_orderitem
        ON restaurant_orderitem.order_id = restaurant_order.id
    INNER JOIN restaurant_product
    ON restaurant_orderitem.product_id = restaurant_product.id
"""
)
columns = [desc[0] for desc in cur.description]
with open("data.csv", "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(columns)
    writer.writerows(res.fetchall())

dt = pd.read_csv("data.csv")
dt

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
grouped = dt.groupby(["name"])[["quantity"]].sum()
top10 = grouped.sort_values("quantity", ascending=False)[:10].copy()
total = top10["quantity"].sum()
top10["quantity"].plot.pie(
    autopct=f"%1.1f%%",
    figsize=(8, 8),
    startangle=90,
    ylabel="",
    title="Top 10 positions in meny by quantity"
)

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
dt["item_total"] = dt["price"] * dt["quantity"]
grouped = dt.groupby("name")[["quantity", "item_total"]].sum().reset_index()
top10 = grouped.sort_values("item_total", ascending=False)[:10].copy().set_index("name")
top10["item_total"].plot.pie(
    autopct="%1.1f%%",
    figsize=(8, 8),
    startangle=90,
    ylabel="",
    title="Top 10 positions in meny by item price"
)

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
dt["datetime"] = pd.to_datetime(dt["datetime"])
dt["hour"] = dt["datetime"].dt.hour
dt.groupby("hour")["item_total"].sum().plot.bar(figsize=(10,5))

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
dt["datetime"] = pd.to_datetime(dt["datetime"])
dt["week_num"] = dt["datetime"].dt.dayofweek
dt.groupby("week_num")["item_total"].sum().plot.bar(figsize=(10,5))